# NamingChecker

1. Фильтрация по классам МКТУ
2. Levenshtein, Jaro-Winkler, нормализованное расстояние
3. Фонетическое сходство (транслитерация)
4. Combined score -> list results

## 1. Data overview

In [1]:
import pandas as pd
import re
import json

df = pd.read_csv("db.csv")
df.head(5)

,#,id,status_fips,order_status_fips,status_fips_date,order_status_fips_date,registration_number,registration_date,expiry_date,application_number,...,update_datetime,doc_name,certificate_link,mark_description,image_url,mark_disclaimer_normal,mark_significant_normal,is_normal,mark_significant_ocr,mark_preprocessed
0,1,396770,1.0,15.0,2023-07-03 00:00:00.000000,2022-06-06 00:00:00.000000,951437.0,2023-07-03 00:00:00.000000,2032-05-06 00:00:00.000000,2022729603,...,2023-07-05 20:06:22.323000,951437,https://new.fips.ru/registers-doc-view/fips_se...,Заявленное обозначение «АГРОЭКО Честное Мясо» ...,https://195.209.177.51/ofpstorage/Doc/TM/RUNWT...,NaN,агроэко честное мясо,True,NaN,агроэко честное мясо
1,2,1449053,NaN,15.0,NaN,2025-03-19 00:00:00.000000,NaN,NaN,NaN,2025721177,...,2025-03-24 19:35:12.751000,2025721177,https://new.fips.ru/registers-doc-view/fips_se...,Заявляемое обозначение представляет собой комп...,https://195.209.177.51/Image/RUTMAP_Images/NEW...,NaN,dr koffer,True,NaN,dr koffer
2,3,1449052,NaN,15.0,NaN,2025-03-18 00:00:00.000000,NaN,NaN,NaN,2025721178,...,2025-03-21 05:06:23.600000,2025721178,https://new.fips.ru/registers-doc-view/fips_se...,На регистрацию заявлено словесное обозначение ...,https://195.209.177.51/Image/RUTMAP_Images/NEW...,NaN,cass,True,NaN,cass
3,4,1161502,NaN,18.0,NaN,2021-01-27 00:00:00.000000,NaN,NaN,NaN,2019758558,...,2025-08-08 14:37:55.466000,2019758558,https://new.fips.ru/registers-doc-view/fips_se...,"Заявляется словесное обозначение, состоящее из...",https://195.209.177.51/Image/RUTMAP_Images/NEW...,NaN,лоза тамани виноградники loza tamani vineyards,True,NaN,лоза тамани виноградники loza tamani vineyards
4,5,765186,1.0,NaN,2003-11-23 00:00:00.000000,NaN,255484.0,2003-09-15 00:00:00.000000,2032-07-16 00:00:00.000000,2002713649,...,2022-10-25 15:32:16.620000,255484,https://new.fips.ru/registers-doc-view/fips_se...,NaN,https://195.209.177.51/Archive/TM/Arc/2008.11....,NaN,artplay,True,NaN,artplay


In [2]:
key_cols = [
    "class_number",
    "mark_significant",
    "mark_disclaimer",
    "mark_significant_normal",
    "mark_preprocessed",
    "certificate_link",
]
df[key_cols].head(10)

,class_number,mark_significant,mark_disclaimer,mark_significant_normal,mark_preprocessed,certificate_link
0,"[""29"",""30"",""31"",""35"",""40""]","[""АГРОЭКО"",""ЧЕСТНОЕ"",""МЯСО""]",Словесный элемент «МЯСО».,агроэко честное мясо,агроэко честное мясо,https://new.fips.ru/registers-doc-view/fips_se...
1,"[""18"",""25"",""35""]","[""Dr. Koffer""]",NaN,dr koffer,dr koffer,https://new.fips.ru/registers-doc-view/fips_se...
2,"[""03"",""35"",""44""]","[""Cass""]",NaN,cass,cass,https://new.fips.ru/registers-doc-view/fips_se...
3,"[""32"",""33""]","[""ЛОЗА ТАМАНИ виноградники LOZA TAMANI vineya...",NaN,лоза тамани виноградники loza tamani vineyards,лоза тамани виноградники loza tamani vineyards,https://new.fips.ru/registers-doc-view/fips_se...
4,"[""11"",""16"",""18"",""20"",""21"",""24"",""27"",""35"",""41"",...","[""ARTPLAY""]",NaN,artplay,artplay,https://new.fips.ru/registers-doc-view/fips_se...
5,"[""16"",""20"",""35"",""41"",""42""]","[""LOOK"",""UP"",""LOOKUP"",""ДИЗАЙН"",""-"",""ЦЕНТР""]",Дизайн-центр.,look up lookup,lookup,https://new.fips.ru/registers-doc-view/fips_se...
6,"[""01"",""02"",""03"",""04"",""05"",""06"",""07"",""08"",""09"",...","[""МАГИЯ НЕЖНОСТИ""]",NaN,магия нежности,магия нежности,https://new.fips.ru/registers-doc-view/fips_se...
7,"[""09"",""41"",""42""]","[""КиберГРП""]",NaN,кибергрп,кибергрп,https://new.fips.ru/registers-doc-view/fips_se...
8,"[""25"",""35""]","[""get go out""]",NaN,get go out,get go out,https://new.fips.ru/registers-doc-view/fips_se...
9,"[""25""]","[""Favourite kids""]",NaN,favourite kids,favourite kids,https://new.fips.ru/registers-doc-view/fips_se...


## 2. Data parsing

In [3]:
def parse_class_numbers(val: str) -> set[int]:
    """Парсинг строки с классами МКТУ в множество чисел."""
    if pd.isna(val):
        return set()
    try:
        return {int(x) for x in json.loads(val)}
    except (json.JSONDecodeError, ValueError, TypeError):
        return set()


df["mktu_classes"] = df["class_number"].apply(parse_class_numbers)

print("Example:")
for i, row in df.head(5).iterrows():
    print(f'\t{row["class_number"]} -> {row["mktu_classes"]}')

Example:
	["29","30","31","35","40"] -> {35, 40, 29, 30, 31}
	["18","25","35"] -> {25, 18, 35}
	["03","35","44"] -> {35, 3, 44}
	["32","33"] -> {32, 33}
	["11","16","18","20","21","24","27","35","41","42"] -> {35, 41, 42, 11, 16, 18, 20, 21, 24, 27}


In [ ]:
# mark_preprocessed как основное, иначе mark_significant_normal
df["name_clean"] = df["mark_preprocessed"].fillna(df["mark_significant_normal"]).fillna("")
df["name_clean"] = df["name_clean"].astype(str).str.strip()

print(f"Всего записей: {len(df)}")
print(f'С непустым именем: {(df["name_clean"] != "").sum()}')
print(f'С пустым именем: {(df["name_clean"] == "").sum()}')

df_valid = df[df["name_clean"] != ""].copy().reset_index(drop=True)
print(f"\nРабочий датасет: {len(df_valid)} записей")

Всего записей: 1000
С непустым именем: 938
С пустым именем: 62

Рабочий датасет: 938 записей


## 3. Text metrics

In [ ]:
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler


def text_similarity_scores(query: str, candidate: str) -> dict:
    q = query.lower().strip()
    c = candidate.lower().strip()

    if not q or not c:
        return {
            "levenshtein": 0.0,
            "jaro_winkler": 0.0,
            "token_sort": 0.0,
            "token_set": 0.0,
            "partial": 0.0,
            "prefix": 0.0,
        }

    prefix_len = next((i for i, (a, b) in enumerate(zip(q, c)) if a != b), min(len(q), len(c)))

    return {
        "levenshtein": Levenshtein.normalized_similarity(q, c),
        "jaro_winkler": JaroWinkler.similarity(q, c),
        "token_sort": fuzz.token_sort_ratio(q, c) / 100.0,
        "token_set": fuzz.token_set_ratio(q, c) / 100.0,
        "partial": fuzz.partial_ratio(q, c) / 100.0,
        "prefix": prefix_len / max(len(q), len(c)),
    }


# Тест
examples = [
    ("EUROPLEX", "EUROFLEX"),
    ("TIGOTA", "DICOTA"),
    ("PROBI", "ПРОБИМАКС"),
    ("ALTO", "ALTA"),
    ("HRBRANDING", "HRBRAND"),
    ("RANGER", "RANGE"),
]

for a, b in examples:
    scores = text_similarity_scores(a, b)
    print(f"{a} vs {b}:")
    for k, v in scores.items():
        print(f"  {k}: {v:.3f}")
    print()

EUROPLEX vs EUROFLEX:
  levenshtein: 0.875
  jaro_winkler: 0.950
  token_sort: 0.875
  token_set: 0.875
  partial: 0.875
  prefix: 0.500

TIGOTA vs DICOTA:
  levenshtein: 0.667
  jaro_winkler: 0.778
  token_sort: 0.667
  token_set: 0.667
  partial: 0.727
  prefix: 0.000

PROBI vs ПРОБИМАКС:
  levenshtein: 0.000
  jaro_winkler: 0.000
  token_sort: 0.000
  token_set: 0.000
  partial: 0.000
  prefix: 0.000

ALTO vs ALTA:
  levenshtein: 0.750
  jaro_winkler: 0.883
  token_sort: 0.750
  token_set: 0.750
  partial: 0.857
  prefix: 0.750

HRBRANDING vs HRBRAND:
  levenshtein: 0.700
  jaro_winkler: 0.940
  token_sort: 0.824
  token_set: 0.824
  partial: 1.000
  prefix: 0.700

RANGER vs RANGE:
  levenshtein: 0.833
  jaro_winkler: 0.967
  token_sort: 0.909
  token_set: 0.909
  partial: 1.000
  prefix: 0.833



## 4. Phonetics

In [ ]:
TRANSLIT_MAP = {
    "а": "a",
    "б": "b",
    "в": "v",
    "г": "g",
    "д": "d",
    "е": "e",
    "ё": "yo",
    "ж": "zh",
    "з": "z",
    "и": "i",
    "й": "y",
    "к": "k",
    "л": "l",
    "м": "m",
    "н": "n",
    "о": "o",
    "п": "p",
    "р": "r",
    "с": "s",
    "т": "t",
    "у": "u",
    "ф": "f",
    "х": "kh",
    "ц": "ts",
    "ч": "ch",
    "ш": "sh",
    "щ": "shch",
    "ъ": "",
    "ы": "y",
    "ь": "",
    "э": "e",
    "ю": "yu",
    "я": "ya",
}

PHONETIC_GROUPS_LATIN = {
    "b": "P",
    "p": "P",
    "d": "T",
    "t": "T",
    "g": "K",
    "k": "K",
    "c": "K",
    "q": "K",
    "v": "F",
    "f": "F",
    "w": "F",
    "z": "S",
    "s": "S",
    "x": "KS",
    "j": "J",
    "l": "L",
    "r": "R",
    "m": "M",
    "n": "N",
    "h": "H",
    "a": "A",
    "e": "E",
    "i": "I",
    "o": "O",
    "u": "U",
    "y": "I",
}


def to_latin_phonetic(text: str) -> str:
    return "".join(TRANSLIT_MAP.get(ch, ch) for ch in text.lower().strip())


def phonetic_code(text: str) -> str:
    code = []
    for ch in re.sub(r"[^a-z]", "", to_latin_phonetic(text)):
        mapped = PHONETIC_GROUPS_LATIN.get(ch, ch.upper())
        if not code or code[-1] != mapped:
            code.append(mapped)
    return "".join(code)


def phonetic_similarity(query: str, candidate: str) -> float:
    q_clean = re.sub(r"[^a-z]", "", to_latin_phonetic(query))
    c_clean = re.sub(r"[^a-z]", "", to_latin_phonetic(candidate))
    if not q_clean or not c_clean:
        return 0.0
    q_code, c_code = phonetic_code(query), phonetic_code(candidate)
    jw_phonetic = JaroWinkler.similarity(q_code, c_code) if q_code and c_code else 0.0
    return 0.6 * JaroWinkler.similarity(q_clean, c_clean) + 0.4 * jw_phonetic

In [ ]:
phonetic_examples = [
    ("EUROPLEX", "EUROFLEX"),
    ("TIGOTA", "DICOTA"),
    ("Cellexin", "СЕЛЕКСЕН"),
    ("ДИКСИКА", "DIXY"),
    ("Реком", "Рекод"),
    ("ALTO", "ALTA"),
    ("PROBI", "ПРОБИМАКС"),
]

headers = ("Original", "Latin Phonetic", "Phonetic Code")
for a, b in phonetic_examples:
    rows = [(a, to_latin_phonetic(a), phonetic_code(a)), (b, to_latin_phonetic(b), phonetic_code(b))]
    widths = [max(len(h), max(len(r[i]) for r in rows)) for i, h in enumerate(headers)]
    sep = "  +-" + "-+-".join("-" * w for w in widths) + "-+"

    print(f"{a} / {b} => Phonetic similarity: {phonetic_similarity(a, b):.3f}")
    print(sep)
    print("  | " + " | ".join(f"{h:<{w}}" for h, w in zip(headers, widths)) + " |")
    print(sep)
    for row in rows:
        print("  | " + " | ".join(f"{v:<{w}}" for v, w in zip(row, widths)) + " |")
    print(sep)
    print()

EUROPLEX / EUROFLEX => Phonetic similarity: 0.952
  +----------+----------------+---------------+
  | Original | Latin Phonetic | Phonetic Code |
  +----------+----------------+---------------+
  | EUROPLEX | europlex       | EUROPLEKS     |
  | EUROFLEX | euroflex       | EUROFLEKS     |
  +----------+----------------+---------------+

TIGOTA / DICOTA => Phonetic similarity: 0.867
  +----------+----------------+---------------+
  | Original | Latin Phonetic | Phonetic Code |
  +----------+----------------+---------------+
  | TIGOTA   | tigota         | TIKOTA        |
  | DICOTA   | dicota         | TIKOTA        |
  +----------+----------------+---------------+

Cellexin / СЕЛЕКСЕН => Phonetic similarity: 0.733
  +----------+----------------+---------------+
  | Original | Latin Phonetic | Phonetic Code |
  +----------+----------------+---------------+
  | Cellexin | cellexin       | KELEKSIN      |
  | СЕЛЕКСЕН | seleksen       | SELEKSEN      |
  +----------+----------------+-----